# Day 1 — Document Ingestion
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 1 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

This notebook walks through the full Day 1 pipeline step by step: parsing a real clinical
guideline PDF, choosing a chunking strategy, generating embeddings, and building a
queryable vector index — the same steps implemented in `ingest.py`, but broken apart here
so you can inspect what happens at each stage before you rely on the script.

**By the end of this notebook you will be able to:**
1. Explain why grounding — not raw model memory — matters in clinical AI
2. Parse a PDF and inspect its extracted structure
3. Compare fixed-size vs. section-aware chunking on the same document
4. Generate an embedding and explain what the resulting vector represents
5. Build a persisted vector index and run a real query against it

> **Data source:** this notebook uses `data/WHO_Hypertension_Guideline_2021.pdf`, the real
> WHO guideline bundled with your starter kit — not a toy example.


## 0. Setup

Run this cell first. It adds the repo root to the path so we can reuse the exact same
functions defined in `ingest.py`, and confirms your environment is ready.


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from pathlib import Path

print("Data directory:", config.DATA_DIR)
print("Chunk size (tokens):", config.CHUNK_SIZE)
print("Chunk overlap (tokens):", config.CHUNK_OVERLAP)
print("PDFs found:", [p.name for p in config.DATA_DIR.glob("*.pdf")])


Data directory: Data
Chunk size (tokens): 1000
Chunk overlap (tokens): 200
PDFs found: ['_OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf']


## 1. Why Grounding Matters

Before touching any code, sit with this for a second: a large language model can generate
a fluent, confident-sounding clinical recommendation **even when it has no real evidence
behind it.** It has no built-in mechanism to say "I don't know."

Retrieval-Augmented Generation (RAG) fixes this by separating two things:

- **What the model knows** (its training data — broad, but unverifiable and possibly stale)
- **What the model is allowed to say** (only what's in the text you hand it right now)

Everything you build today is the **first half** of that separation: turning a trustworthy
PDF into a searchable, citable index. Day 3 builds the second half (forcing the model to
answer only from what this index returns).


## 2. Step 1 — Parse the PDF

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page, each carrying
page-level metadata automatically (page number, source path).

Run the cell below and inspect the output. Notice that `page.metadata["page"]` is
**zero-indexed** — page index `0` is the PDF's first page.


In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(config.DATA_DIR.glob("*.pdf"))[0]
print(f"Loading: {pdf_path.name}\n")

loader = PyPDFLoader(str(pdf_path))
raw_pages = loader.load()

print(f"Loaded {len(raw_pages)} pages.\n")
print("--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---")
print(raw_pages[2].metadata)
print("\n--- Page 3 (index 2) first 400 characters ---")
print(raw_pages[2].page_content[:400])


/tmp/ipykernel_1592/1481711733.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading: _OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf

Loaded 312 pages.

--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---
{'producer': 'Internet Archive PDF 1.4.16; including mupdf and pymupdf/skimage', 'creator': 'Internet Archive (Scribe Version 5.3-initial-170-ga5e5737d)', 'creationdate': '2022-11-22T19:22:16+00:00', 'title': "Harrison's Principles of internal medicine. Update", 'author': 'NullObject', 'moddate': '2022-11-22T19:22:16+00:00', 'subject': 'Internal medicine; Internal medicine -- Examinations, questions, etc', 'keywords': 'https://archive.org/details/harrisonsprincip0000unse_w1j5_9ed', 'trapped': 'NullObject', 'source': 'Data/_OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf', 'total_pages': 312, 'page': 2, 'page_label': ''}

--- Page 3 (index 2) first 400 characters ---



### Raw metadata isn't citation-ready yet

Look at the metadata above: PyPDFLoader gives you a generic `page` index and a full file
`source` path — but nothing called `document_name`, and no human-friendly 1-indexed page
number. If we chunk these pages as-is, every citation later would say "unknown, page ?".

`load_pdfs()` in `ingest.py` does one small but critical thing: it stamps
`document_name` and a 1-indexed `page_number` onto every page's metadata **before**
chunking, so that metadata survives all the way through to the final citation.


In [3]:
from ingest import load_pdfs

pages = load_pdfs(config.DATA_DIR)

print("--- Page 3 (index 2) metadata AFTER normalization ---")
print({k: pages[2].metadata[k] for k in ["document_name", "page_number", "page"]})


/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Harrisons 
Isselbacher - Adams Braunwald Pp 3 ersd




Digitized by the Internet Archive 
In 2022 with funding from 
Kahle/Austin Foundation 
https://archive.org/details/harrisonsprincipO000unse_w1j5 9ed
UPDATE | 
Harrison's 
PRINCIPLES 
OF INTERNAL 
MEDICINE 2 
with CME Examination
ARTICLES FORTHCOMING IN UPDATE II 
Dietary Fiber in Intestinal Disorders Mark S. McPhee 
Thrombosis and Antithrombotic Therapy Daniel Deykin 
Immune Granulocytopenia and Thrombocytopenia 
Gerald L. Logue and Wendell F. Rosse 
Radiologic Imaging of the Pancreas: 1981 
Joseph T. Ferrucci, Jr., and Jack Wittenberg 
Mechanisms and Management of Pain Howard Fields 
Treatable Forms of Dementia Charles E. Wells 
Polymyositis: Diagnosis and Management Carl Pearson 
The Masked Depression Brian Woods and Raymond D. Adams 
Limitation of Myocardial Infarct Size Eugene Braunwald 
Ventricular Premature Beats: Why, When, and How to Treat 
Bernard Lown and Philip J. Podrid 
Vasodilators and Newer Inotropic Agents in the Tr

In [8]:
print(pages[24].page_content[:400])

ducts), and PTHC is more widely available at 
somewhat lower cost. Added advantages include 
the ability to delineate the proximal biliary tree in 
an obstructed system (which may be important 
preoperatively), the ability to outline the biliary 
tree following a Roux-en-Y diversion with bil- 
iary-enteric anastomosis (where ERCP is essen- 
tially impossible), and the capability for separate 
demo


### Checkpoint 1

Look at the printed text above. Answer for yourself before moving on:

- Are section headings ("3.1 Blood pressure threshold...") visible as recognizable text, or
  did they get mangled?
- Are there any obvious parsing artifacts (broken words, merged columns, stray characters)?

If parsing looks clean here, section-aware chunking (Step 2) will work well. If it looks
messy, no chunking strategy will fully save you — the fix belongs upstream, in parsing.


## 3. Step 2 — Compare Chunking Strategies

We'll build **two** chunkers on the same pages and compare them directly: a naive
fixed-size splitter, and the section-aware splitter actually used in `ingest.py`.


In [4]:
#!-------------------------------!
# NOTE: You can use whatever Text splitter you prefer, e.g. (NLTK, spaCy)
#!-------------------------------!

from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Naive fixed-size splitter: no regard for sentence/paragraph boundaries ---
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # characters, not tokens — deliberately crude
    chunk_overlap=0,
    separators=[""],       # forces raw character-count splitting
)
naive_chunks = naive_splitter.split_documents(pages)

# --- Section-aware splitter: same one used in ingest.py ---
aware_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE * 4,      # ~4 chars/token estimate
    chunk_overlap=config.CHUNK_OVERLAP * 4,
    separators=["\n\n", "\n", ". ", " ", ""],
)
aware_chunks = aware_splitter.split_documents(pages)

print(f"Naive fixed-size chunker:   {len(naive_chunks)} chunks")
print(f"Section-aware chunker:     {len(aware_chunks)} chunks")


Naive fixed-size chunker:   5619 chunks
Section-aware chunker:     453 chunks


In [5]:
# Look at one naive chunk boundary — notice it can cut mid-sentence
print("--- Naive chunk #5 (often cuts mid-sentence) ---")
print(repr(naive_chunks[5].page_content))

print("\n--- Section-aware chunk #5 (respects paragraph breaks) ---")
print(repr(aware_chunks[5].page_content[:300]))


--- Naive chunk #5 (often cuts mid-sentence) ---
'. Wells \nPolymyositis: Diagnosis and Management Carl Pearson \nThe Masked Depression Brian Woods and Raymond D. Adams \nLimitation of Myocardial Infarct Size Eugene Braunwald \nVentricular Premature Beat'

--- Section-aware chunk #5 (respects paragraph breaks) ---
'NOTICE \nMedicine is an ever-changing science. As new \nresearch and clinical experience broaden our \nknowledge, changes in treatment and drug \ntherapy are required. The editors and the \npublisher of this work have made every effort to \nensure that the drug dosage schedules herein are \naccurate and in'


### Checkpoint 2

Compare the two printed chunks above.

- Does the naive chunk end mid-word or mid-sentence?
- Does the section-aware chunk end at a more natural paragraph or sentence boundary?

This is the entire argument for section-aware chunking in one comparison: **the boundary
you cut at becomes the boundary a citation has to point to.** A citation that lands mid-sentence
is much harder for a clinician to trust and verify.


## 4. Step 3 — Attach Citation Metadata

A chunk without a traceable source is useless for a clinical tool. Before embedding
anything, every chunk needs: **document name, page number, and a stable chunk id.**
This is exactly what `chunk_documents()` in `ingest.py` does — let's call it directly.


In [6]:
from ingest import chunk_documents

chunks = chunk_documents(pages)
print(f"Total chunks with metadata attached: {len(chunks)}\n")

sample = chunks[10]
print("--- Sample chunk metadata ---")
for k in ["document_name", "page_number", "chunk_id"]:
    print(f"  {k}: {sample.metadata.get(k)}")
print("\n--- Sample chunk text ---")
print(sample.page_content[:300])


Total chunks with metadata attached: 453

--- Sample chunk metadata ---
  document_name: _OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf
  page_number: 14
  chunk_id: None

--- Sample chunk text ---
SSS 
SSS SSD 
PREFACE 
Updates to accompany Harrison’s Principles of 
Internal Medicine? Why, what, and how often? 
The editors of Harrison’s have struggled, usu- 
ally unsuccessfully, to keep each new edition 
from expanding. Despite admonitions to them- 
selves and to the authors, these efforts ha


## 5. Step 4 — What Is an Embedding, Really?

An embedding model converts text into a list of numbers (a **vector**) that captures
meaning — texts about similar topics end up as vectors that point in similar directions,
even if they don't share any of the same words.

Let's embed three short phrases and check which two are "closer" in vector space.


In [7]:
import numpy as np
from ingest import get_embedding_function

embed_fn = get_embedding_function()

texts = [
    "first-line treatment for hypertension",
    "initial therapy for high blood pressure",   # means the same thing, different words
    "recommended screening interval for breast cancer",  # unrelated topic
]

vectors = embed_fn.embed_documents(texts)
vectors = np.array(vectors)
print(f"Each embedding is a vector of length {vectors.shape[1]}\n")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_related = cosine_similarity(vectors[0], vectors[1])
sim_unrelated = cosine_similarity(vectors[0], vectors[2])

print(f"Similarity — same meaning, different words:  {sim_related:.3f}")
print(f"Similarity — genuinely different topics:      {sim_unrelated:.3f}")


Fetching 5 files: 100%|██████████| 5/5 [00:11<00:00,  2.39s/it]


Each embedding is a vector of length 384

Similarity — same meaning, different words:  0.855
Similarity — genuinely different topics:      0.578


### Checkpoint 3

You should see the first similarity score noticeably **higher** than the second — the
"same meaning, different words" pair should score closer together, even though they share
almost no words in common. This is the entire mechanism semantic search relies on. If both
scores came out similar, something about the embedding model would be worth investigating
before trusting it on Day 2.


## 6. Step 5 — Build the Vector Index

Now we embed every chunk and store it in a local ChromaDB collection, using the exact
same `build_index()` function from `ingest.py`. This is the same call the script makes —
seeing it run here just makes the process visible.

> First run downloads a small local embedding model (~100MB) — this happens once and is
> cached afterward.


In [8]:
from ingest import build_index

vectordb = build_index(chunks)
print("\nIndex build complete.")



Index build complete.


## 7. Step 6 — Run a Real Query

The whole point of everything above: ask a real clinical question and see whether the
index returns something relevant.


In [9]:
questions = [
    "what is the single most useful noninvasive procedure for differentiating intrahepatic cholestasis from extrahepatic obstruction?",
    "Describe the mechanism by which HbA1c (glycosylated hemoglobin) provides a measure of long-term diabetic control. How is it formed, and what does its level reflect?",
    "What is the most effective treatment for severe hypersomnia sleep-apnea syndrome (HSA), and what clinical improvements are observed after this intervention?",
    "A 68-year-old woman with hypertension presents with dizziness, vomiting, and inability to stand. She has severe truncal ataxia but minimal appendicular ataxia. Two days after admission, she develops dysarthria and progressive obtundation. Vertebral angiography reveals an embolic occlusion in the distal posterior inferior cerebellar artery. What is the most likely cause of her clinical deterioration, and what treatment would be most appropriate?",
    "What is the most common etiologic agent causing severe diarrhea in children between 6 and 18 months of age worldwide?",
    "Why is oral vancomycin the preferred treatment for Clostridium difficile-associated pseudomembranous colitis rather than intravenous administration, and what is the recommended dosage?",
    'What is the primary advantage of using a programmable open-loop insulin delivery system compared to a closed-loop ("artificial pancreas") system for long-term management of diabetes? What limitation does the open-loop system have compared to the closed-loop system?',
    "According to the Royal College of General Practitioners (RCGP) study on complications of the birth control pill, what was the cardiovascular mortality rate ratio for oral contraceptive users compared to controls for those who used the pill for 1 to 59 months, and what was it for those who used it for 60 months or more?",
    "What is the most reliable diagnostic test for meningeal carcinomatosis, and what CSF findings are typically associated with this condition?",
    "A patient with a history of cancer and hypercalcemia is found to have elevated urinary prostaglandin E metabolites. The patient responds to indomethacin therapy with normalization of serum calcium. Explain the mechanism of hypercalcemia in this patient, including the role of prostaglandin E and the therapeutic rationale for using indomethacin."
]
answers = [
    "it is evident that ultrasonography is the single most useful noninvasive procedure in the differentiation of intrahepatic cholestasis from extrahepatic obstruction. The clinical information gained from the ultrasound usually represents the basis for the selection of subsequent diagnostic tests.",
    "these findings indicate that glycosylation of hemoglobin takes place slowly and continuously throughout the 120- day life span of the red cell... The measurement of Hb A1c in diabetics provides a useful index of cumulative control of hyperglycemia during the 2 to 3 months immediately prior to the measurement.",
    "The most effective treatment of severe HSA is to provide the patient with an open tracheal airway (tracheostomy) during the night's sleep. Tracheostomy results in marked reduction in daytime sleepiness and improvement in mood, a decrease in systemic hypertension, a fall in an abnormally high hematocrit, elimination of the bradycardia-tachycardia pattern, and in most cases a marked decrease in the frequency of sleep-related cardiac arrhythmias.",
    "The posterior inferior parts of the cerebellum are supplied by the posterior inferior cerebellar artery and may also become infarcted if collateral circulation is not provided by the anterior inferior cerebellar artery. Rarely, edema formation in the infarcted cerebellum may cause pressure symptoms in the posterior fossa and result in progressive obtundation and death... surgical decompression may be necessary.",
    "Rotavirus was first established as an etiology of infantile diarrhea in 1973 in Australia. Since then it has been found to have a worldwide distribution and to be one of the most common causes of severe diarrhea in children between the ages of 6 to 18 months.",
    "The patient should be started on oral vancomycin 500 mg every 6 h. Parenteral antibiotic is not indicated in this disease, since the organisms are not invasive but are growing within the lumen of the colon.",
    "Open- loop systems sacrifice the precision and self- adaptation of the closed- loop instruments but do supply insulin at varying predetermined rates... Because these systems are portable, unrestrained subjects can participate in studies of longer duration.",
    "In the RCGP study, the cardiovascular mortality rate was 3.4 times that of controls among those who used oral contraceptives for 1 to 59 months and 9.7 times that of controls for those using them for 60 months or more.",
    "Tumor cells were found in approximately 50 percent of our cases, providing the fluid was cytocentrifuged or filtered on a millipore filter. Hemocytometer count and a differential analysis of cells in the counting chamber are unreliable. Cytocentrifugation of a few milliliters of fluid will often yield several hundred cells even when the hemocytometer count is zero... CSF pressure was elevated in many cases... Glucose (≤ 45 mg/100 ml) 40%, Protein (≥ 50 mg/100 ml) 70%, Cells (≥ 3 mm-3) 18%.",
    "Prostaglandin E2 is a potent inducer of bone resorption and calcium release from bone... Drug trials were performed to ascertain whether agents that inhibit endogenous prostaglandin synthesis also decrease circulating calcium levels. Such drugs were effective in some but not all patients... Patients who have no evidence for elevated PGE production fail to respond... a small subset of patients with hypercalcemia and malignancy, perhaps 10 to 15 percent, have elevated PGE production and can be treated with drugs that inhibit prostaglandin synthesis."
]

In [10]:
# Query using ChromaDB's native methods


# Get results from ChromaDB
results = vectordb.query(
    query_texts=[questions[0]],  # just the first question for now
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# Print results
print(f"Question: {questions[0]}\n")
for i in range(len(results['ids'][0])):
    doc = results['documents'][0][i]
    metadata = results['metadatas'][0][i]
    distance = results['distances'][0][i]
    # Convert distance to similarity score (optional)
    score = 1 / (1 + distance)  # Simple conversion
    
    print(f"[{i+1}] score={score:.3f}  {metadata.get('document_name')}, "
          f"page {metadata.get('page_number')}")
    print(f"    \"{doc[:180].strip()}...\"\n")

Question: what is the single most useful noninvasive procedure for differentiating intrahepatic cholestasis from extrahepatic obstruction?

[1] score=0.576  _OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf, page 39
    "22 BILIARY OBSTRUCTION: CURRENT APPROACHES TO DIAGNOSIS AND TREATMENT 
astomosis, pancreatoduodenectomy, total pan- 
createctomy, endoscopic sphincterotomy, or 
percutaneous stone..."

[2] score=0.570  _OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf, page 288
    "CME EXAMINATION 
BILIARY OBSTRUCTION: CURRENT 
APPROACHES TO DIAGNOSIS AND 
TREATMENT 
Questions 1 to 3 
A 78-year-old man is admitted to the hosital 
because of gradually deepenin..."

[3] score=0.568  _OceanofPDF.com_Harrisons_principals_of_internal_medicine_-_Tinsley_R_harrison.pdf, page 29
    "BILIARY OBSTRUCTION: CURRENT APPROACHES TO DIAGNOSIS AND TREATMENT 
patients requiring rapid, accurate diagnosis and 
therapy. Of course, laparoto

### Checkpoint 4 — Day 1 Self-Check

Before you close this notebook, confirm all of the following are true:

- [ ] The top retrieved chunk in Step 6 is genuinely relevant to the question asked
- [ ] Every result shows a document name and page number (not `None`)
- [ ] You could explain to a teammate, in one sentence, why section-aware chunking beat
      the naive splitter in Checkpoint 2

If any of these aren't true yet, that's normal — go back to the relevant step above and
adjust `config.py` (chunk size, overlap) before moving on to Day 2.

## What's Next

Day 2's notebook picks up exactly here: tuning `top_k`, benchmarking this embedding model
against alternatives, and proving your retrieval quality with real, logged numbers instead
of a single example query.


In [ ]:
import numpy as np

def evaluate_retrieval(query, results, answers_list, questions_list, k_values=[1, 3, 5]):
    retrieved_ids = results['ids'][0]
    distances = results['distances'][0]
    
    cosine_sims = [1 - d for d in distances]
    
    print(f"Query: {query}\n")
    print(f"{'Rank':<6} {'Cosine Sim':<12} {'Distance':<10} {'Document ID'}")
    print("-" * 60)
    
    for i in range(len(retrieved_ids)):
        doc_id = retrieved_ids[i]
        print(f"{i+1:<6} {cosine_sims[i]:.4f}     {distances[i]:.4f}     {doc_id[:30]}...")
    
    query_index = None
    for idx, q in enumerate(questions_list):
        if q == query:
            query_index = idx
            break
    
    if query_index is not None and query_index < len(answers_list):
        relevant_answer = answers_list[query_index]
        print(f"\nRelevant answer (first 200 chars): {relevant_answer[:200]}...\n")
        
        retrieved_docs = results['documents'][0]

        relevant_doc_indices = []
        
        if 'metadatas' in results and results['metadatas'][0]:
            for i, metadata in enumerate(results['metadatas'][0]):
                if 'answer_index' in metadata and metadata['answer_index'] == query_index:
                    relevant_doc_indices.append(i)
                elif 'document_name' in metadata and str(query_index) in metadata['document_name']:
                    relevant_doc_indices.append(i)
        
        if relevant_doc_indices:
            print("=" * 60)
            print("PRECISION@K AGAINST RELEVANT ANSWERS")
            print("=" * 60)
            
            for k in k_values:
                if k <= len(retrieved_ids):
                    hits = sum(1 for i in range(k) if i in relevant_doc_indices)
                    precision = hits / k
                    recall = hits / len(relevant_doc_indices) if relevant_doc_indices else 0
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                    
                    print(f"Precision@{k}: {precision:.3f} ({hits}/{k})")
                    print(f"Recall@{k}:    {recall:.3f}")
                    print(f"F1@{k}:        {f1:.3f}")
                    print("-" * 40)
            
            for i, doc_idx in enumerate(retrieved_ids):
                if i in relevant_doc_indices:
                    print(f"\nMRR: {1/(i+1):.3f} (first relevant answer at rank {i+1})")
                    break
        else:
            print("\n⚠️  No relevant documents found in retrieval results")
            print("Make sure your documents have proper metadata linking them to answers.")
    else:
        print("\n⚠️  Query not found in questions list or no corresponding answer available.")

results = vectordb.query(
    query_texts=[questions[0]],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

evaluate_retrieval(questions[0], results, answers, questions, k_values=[1, 3, 5])

Query: what is the single most useful noninvasive procedure for differentiating intrahepatic cholestasis from extrahepatic obstruction?

Rank   Cosine Sim   Distance   Document ID
------------------------------------------------------------
1      0.2632     0.7368     _OceanofPDF.com_Harrisons_prin...
2      0.2450     0.7550     _OceanofPDF.com_Harrisons_prin...
3      0.2384     0.7616     _OceanofPDF.com_Harrisons_prin...
4      0.2334     0.7666     _OceanofPDF.com_Harrisons_prin...
5      0.2220     0.7780     _OceanofPDF.com_Harrisons_prin...

Relevant answer (first 200 chars): it is evident that ultrasonography is the single most useful noninvasive procedure in the differentiation of intrahepatic cholestasis from extrahepatic obstruction. The clinical information gained fro...


⚠️  No relevant documents found in retrieval results
Make sure your documents have proper metadata linking them to answers.
